In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../data/ssd_data.csv")
df.head()

,temperature,wear_level,power_on_hours,write_cycles,ECC_error_rate,bad_blocks,RUL,health_status
0,50.0,3.2,15633,97789,0.0168,1,9408,Healthy
1,43.6,46.9,44233,20058,0.0232,2,5725,Warning
2,51.5,92.5,44967,82807,0.0012,3,1858,Critical
3,60.2,39.8,1416,31604,0.0162,1,6506,Healthy
4,42.7,62.5,38072,99102,0.0113,1,4783,Warning


In [4]:
def compute_health_index(row):

    temp_score  = 1 - (row["temperature"] - 20) / 60
    wear_score  = 1 - row["wear_level"] / 100
    ecc_score   = 1 - min(row["ECC_error_rate"] / 0.2, 1)
    block_score = 1 - min(row["bad_blocks"] / 20, 1)
    rul_score   = min(row["RUL"] / 10000, 1)

    score = (
        0.15 * temp_score +
        0.25 * wear_score +
        0.25 * ecc_score +
        0.15 * block_score +
        0.20 * rul_score
    )

    return round(score * 100, 1)

In [ ]:
# EXPLANATION
Wear Level and ECC Error Rate were assigned the highest weights (25% each) because they are directly related to flash memory degradation and data reliability.

RUL contributes 20% because it provides an overall estimate of remaining device life.

Temperature and Bad Blocks contribute 15% each since they influence SSD health but are typically secondary indicators compared to wear and error rates.

All weights sum to 1.0, ensuring the final score remains properly normalised.

In [5]:
df["health_index"] = df.apply(compute_health_index, axis=1)

df.head()

,temperature,wear_level,power_on_hours,write_cycles,ECC_error_rate,bad_blocks,RUL,health_status,health_index
0,50.0,3.2,15633,97789,0.0168,1,9408,Healthy,87.7
1,43.6,46.9,44233,20058,0.0232,2,5725,Warning,69.4
2,51.5,92.5,44967,82807,0.0012,3,1858,Critical,50.3
3,60.2,39.8,1416,31604,0.0162,1,6506,Healthy,70.2
4,42.7,62.5,38072,99102,0.0113,1,4783,Warning,66.1


In [6]:
print(
    df.groupby("health_status")["health_index"]
      .describe()
      .round(1)
)

                count  mean  std   min   25%   50%   75%   max
health_status                                                 
Critical        480.0  48.7  4.0  33.0  46.5  48.9  51.6  59.3
Healthy        3542.0  79.8  6.7  49.0  74.9  79.8  84.8  97.8
Warning        3978.0  60.4  6.9  36.0  55.2  60.4  65.6  78.8


In [ ]:
# EXPLANATION
The average Health Index increases consistently from Critical to Warning to Healthy devices.

Critical SSDs have the lowest average scores, indicating reduced remaining life and higher degradation.

Warning SSDs occupy the middle range and may require monitoring or preventive maintenance.

Healthy SSDs achieve the highest scores, reflecting lower wear levels and longer expected remaining life.

Although some overlap exists between groups, the overall ordering confirms that the Health Index captures meaningful differences in device condition.

In [ ]:
# EXPLANATION ( Summary of this notebook )
The average Health Index increases consistently from Critical to Warning to Healthy devices.

Critical SSDs have the lowest average scores, indicating reduced remaining life and higher degradation.

Warning SSDs occupy the middle range and may require monitoring or preventive maintenance.

Healthy SSDs achieve the highest scores, reflecting lower wear levels and longer expected remaining life.

Although some overlap exists between groups, the overall ordering confirms that the Health Index captures meaningful differences in device condition.